# Construcción de un Cubo Resumen de Aseguramiento de Ingresos de Telecomunicaciones con PROC SUMMARY

## Resumen Ejecutivo

Un equipo de aseguramiento de ingresos de un operador móvil pre-agrega un mes de registros de facturación diarios por suscriptor en un cubo resumen compacto para que los analistas puedan explorar los ingresos liquidados por región, nivel de plan y tipo de llamada sin volver a recorrer la tabla de hechos original. `PROC SUMMARY` consolida 100 registros de suscriptor-día en un conjunto multidimensional de celdas: el grano más fino (región x nivel de plan x tipo de llamada) llena 25 de las 27 celdas posibles, mientras que los subcubos con nombre ofrecen los totales marginales que los analistas consultan con más frecuencia. En este mes de muestra el operador liquidó \$222.78 en las tres regiones activas, con Sur (\$97.44) y Este (\$86.94) representando juntas el 83% del ingreso y Norte quedando atrás con \$38.40. El subcubo individual más rico es el servicio de Voz de nivel Plus (\$59.06 en 18 registros), y clasificar las celdas por rendimiento por registro revela que las celdas de datos móviles son los objetivos de mayor valor para una auditoría de fugas (rendimiento máximo \$7.87/registro). Cada cifra a continuación se lee directamente de la salida ejecutada.

## Fuentes de Datos

| Conjunto de Datos | Filas | Grano | Variables Clave |
|---------|------|-------|---------------|
| `work.cdr_billing` | 100 | Una fila por resumen de uso de suscriptor-día | `region` (Este/Sur/Norte), `plan_tier` (Prepago/Básico/Plus), `call_type` (Voz/SMS/Datos), `device_class`, `bill_month`, `revenue`, `call_minutes`, `data_mb`, `subscriber_wt` |
| `work.cube_nway` | 25 | Una fila por celda no vacía (región x plan_tier x call_type) | `_FREQ_`, `rev_total`, `rev_mean`, `rev_max`, `min_total`, `data_total` |
| `work.cube_hier` | 22 | Una fila por celda de los subcubos de exploración con nombre | `_TYPE_`, `_FREQ_`, `rev_total` |

Todos los datos se generan en línea con `call streaminit()` / `rand()`; no se requieren archivos externos ni acceso a la red. Este entorno se ejecuta sin licencia, por lo que los conjuntos de datos escritos están limitados a 100 observaciones — el escenario está dimensionado para que el cubo quede completamente poblado dentro de ese límite.

## De registros detallados de llamadas a un cubo explorable

Los operadores móviles liquidan ingresos a partir de millones de eventos de uso diarios. Para que los analistas de aseguramiento de ingresos puedan responder preguntas como *"¿Cuál fue el ingreso de voz de nivel Plus en la región Sur el mes pasado?"* sin volver a recorrer la tabla de hechos original cada vez, **pre-agregamos** los datos en un cubo resumen compacto.

`PROC SUMMARY` es la herramienta de trabajo de Base SAS exactamente para esto: agrupa una tabla de hechos plana por una o más dimensiones `CLASS` y escribe las estadísticas solicitadas en un conjunto de datos de salida, etiquetando cada fila con `_TYPE_` (qué dimensiones están activas) y `_FREQ_` (registros detrás de la celda). Ese conjunto de datos de salida *es* un cubo multidimensional — la misma estructura de consolidación que expondría una herramienta OLAP, materializada como un conjunto de datos SAS ordinario que se puede imprimir, unir y explorar.

Este cuaderno:

1. Genera un mes realista de registros de facturación de suscriptor-día.
2. Construye el cubo de grano más fino (región x nivel de plan x tipo de llamada) con `PROC SUMMARY NWAY`.
3. Materializa **subcubos de exploración** con nombre usando la instrucción `TYPES`.
4. Proyecta el ingreso a la base de suscriptores con un `WEIGHT`, explora una región y clasifica las celdas por rendimiento por registro para el triage de fugas.

## Paso 1 - Generar datos sintéticos de facturación / detalle de llamadas

Cada fila resume el uso de un suscriptor de un servicio en un día. Usamos `call streaminit` para reproducibilidad y `rand()` para extraer distribuciones plausibles: el ingreso y el uso escalan con el nivel de plan, el ingreso de voz sigue los minutos facturables y el ingreso de datos sigue los megabytes. Cada `RAND('table', ...)` recibe una probabilidad por categoría para que cada región, nivel y tipo de llamada aparezca en la muestra de 100 registros. Se adjunta un pequeño peso de encuesta `subscriber_wt` para poder demostrar una medida ponderada más adelante.

In [1]:
DATOS work.cdr_billing;
    LLAMAR streaminit(20260131);
    LONGITUD region $6 plan_tier $9 call_type $7 device_class $13 bill_month $7;
    ETIQUETA region        = "Región"
          plan_tier     = "Nivel de Plan"
          call_type     = "Tipo de Llamada"
          device_class  = "Tipo de Dispositivo"
          bill_month    = "Mes de Facturación"
          revenue       = "Ingreso Liquidado (USD)"
          call_minutes  = "Minutos de Voz Facturables"
          data_mb       = "Volumen de Datos (MB)"
          subscriber_wt = "Peso de Encuesta del Suscriptor";

    HACER i = 1 HASTA 100;
        /* --- Dimensiones: una probabilidad por nivel, que suman 1.0 --- */
        r = rand("table", 0.40, 0.33, 0.27);
        SELECCIONAR (r);
            CUANDO (1) region = "Este";
            CUANDO (2) region = "Sur";
            OTRO region = "Norte";
        END;

        p = rand("table", 0.30, 0.40, 0.30);
        SELECCIONAR (p);
            CUANDO (1) plan_tier = "Prepago";
            CUANDO (2) plan_tier = "Básico";
            OTRO plan_tier = "Plus";
        END;

        c = rand("table", 0.50, 0.30, 0.20);
        SELECCIONAR (c);
            CUANDO (1) call_type = "Voz";
            CUANDO (2) call_type = "SMS";
            OTRO call_type = "Datos";
        END;

        SI rand("uniform") < 0.55 ENTONCES device_class = "Inteligente";
        SINO device_class = "Convencional";

        bill_month = "2026-01";

        /* --- Medidas, escaladas por nivel y servicio --- */
        tier_mult = (plan_tier = "Prepago")*0.7
                  + (plan_tier = "Básico")*1.0
                  + (plan_tier = "Plus")*1.7;

        call_minutes = round((call_type = "Voz")
                       * rand("gamma", 2.0) * 18 * tier_mult, 0.1);
        data_mb      = round((call_type = "Datos")
                       * rand("gamma", 1.5) * 220 * tier_mult, 0.1);

        base_rev = 0.05*call_minutes + 0.010*data_mb
                 + (call_type = "SMS") * rand("poisson", 30) * 0.03;
        revenue = round(base_rev * (0.85 + 0.30*rand("uniform")), 0.01);

        subscriber_wt = round(0.8 + rand("uniform")*1.6, 0.01);

        SALIDA;
    END;
    ELIMINAR i r p c tier_mult base_rev;
EJECUTAR;

PROC PRINT DATOS=work.cdr_billing(obs=8) ETIQUETA noobs;
    TÍTULO "Muestra de Registros de Facturación por Suscriptor-Día";
EJECUTAR;


                                 Muestra de Registros de Facturación por Suscriptor-Día                                 

 Región  Nivel de Plan  Tipo de Llamada  Tipo de Dispositivo   Mes de Facturación  Minutos de Voz Facturables  Volumen de Datos (MB)  Ingreso Liquidado (USD)  Peso de Encuesta del Suscriptor
Norte    Plus           SMS              Inteligente          2026-01                                       0                      0                     0.67                             1.13
Sur      Prepago        SMS              Convencional         2026-01                                       0                      0                     0.94                             1.42
Sur      Plus           SMS              Inteligente          2026-01                                       0                      0                     0.99                             0.86
Sur      Plus           SMS              Inteligente          2026-01                                       0     


NOTE: DATA work.cdr_billing


NOTE: Wrote work.cdr_billing (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.03 seconds
  cpu   0.03 seconds
NOTE: PROC PRINT data=work.cdr_billing

NOTE: PROC PRINT completed: 8 observations printed, 9 variables


## Paso 2 - Construir el cubo de grano más fino con PROC SUMMARY NWAY

`NWAY` conserva solo la combinación más detallada de todas las dimensiones `CLASS` - aquí cada celda poblada (región x plan_tier x call_type). Para cada celda almacenamos la `SUM`, `MEAN` y `MAX` del ingreso más los minutos y megabytes totales, de modo que un analista pueda leer el ingreso total por celda, derivar el promedio por registro y detectar el cargo individual más grande. `_FREQ_` registra cuántos suscriptor-días respaldan cada celda. Aquí eliminamos `_TYPE_` porque, en el grano NWAY, cada fila tiene el mismo tipo.

In [2]:
PROC summary DATOS=work.cdr_billing NWAY;
    CLASE region plan_tier call_type;
    VAR revenue call_minutes data_mb;
    SALIDA out=work.cube_nway(ELIMINAR=_type_)
           sum(revenue)=rev_total  mean(revenue)=rev_mean  MAX(revenue)=rev_max
           sum(call_minutes)=min_total
           sum(data_mb)=data_total;
EJECUTAR;

PROC PRINT DATOS=work.cube_nway(obs=12) ETIQUETA noobs;
    ETIQUETA region     = "Región"
          plan_tier  = "Nivel de Plan"
          call_type  = "Tipo de Llamada"
          rev_total  = "Ingreso Total"
          rev_mean   = "Ingreso Promedio"
          rev_max    = "Ingreso Máximo"
          min_total  = "Minutos Totales"
          data_total = "Datos Totales (MB)";
    TÍTULO "Celdas del cubo NWAY (region * plan_tier * call_type)";
    FORMATO rev_total rev_mean rev_max min_total data_total comma10.2;
EJECUTAR;


                                 Celdas del cubo NWAY (region * plan_tier * call_type)                                  

 Región  Nivel de Plan  Tipo de Llamada  _FREQ_  Ingreso Total  Ingreso Promedio   Ingreso Máximo  Minutos Totales  Datos Totales (MB)
Este     Básico         Datos                 4          18.17              4.54             8.05             0.00            1,807.90
Este     Básico         SMS                   4           4.07              1.02             1.24             0.00                0.00
Este     Básico         Voz                   7          15.08              2.15             3.73           302.50                0.00
Este     Plus           Datos                 1           5.54              5.54             5.54             0.00              573.90
Este     Plus           SMS                   4           3.59              0.90             0.95             0.00                0.00
Este     Plus           Voz                   7          23.87      


NOTE: PROC MEANS
NOTE: Output dataset work.cube_nway has 25 observations and 9 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 12 observations printed, 9 variables


## Paso 3 - Materializar subcubos de exploración con nombre usando TYPES

Un cubo OLAP pre-almacena las consolidaciones que los analistas navegan con más frecuencia. La instrucción `TYPES` hace exactamente eso: cada término le pide a `PROC SUMMARY` que emita un subcubo. Solicitamos el total general `()`, el marginal de `region` y los subcubos de dos vías `region*plan_tier` y `call_type*plan_tier` - las rutas de exploración que expondría un panel de ingresos. SAS etiqueta cada subcubo con un código `_TYPE_` (una máscara de bits sobre la lista `CLASS`), de modo que un único conjunto de datos de salida contiene cada nivel de la jerarquía.

In [3]:
PROC summary DATOS=work.cdr_billing;
    CLASE region plan_tier call_type;
    VAR revenue;
    TYPES () region region*plan_tier call_type*plan_tier;
    SALIDA out=work.cube_hier
           sum(revenue)=rev_total;
EJECUTAR;

PROC PRINT DATOS=work.cube_hier ETIQUETA noobs;
    ETIQUETA region    = "Región"
          plan_tier = "Nivel de Plan"
          call_type = "Tipo de Llamada"
          rev_total = "Ingreso Total";
    TÍTULO "Consolidados del subcubo: total general, region, region*nivel, call_type*nivel";
    VAR _type_ region plan_tier call_type _freq_ rev_total;
    FORMATO rev_total comma10.2;
EJECUTAR;


                     Consolidados del subcubo: total general, region, region*nivel, call_type*nivel                     

_TYPE_   Región  Nivel de Plan  Tipo de Llamada  _FREQ_  Ingreso Total
     0                                              100         222.78
     3           Básico         Datos                 8          38.06
     3           Básico         SMS                   8           8.03
     3           Básico         Voz                  20          42.33
     3           Plus           Datos                 3          17.46
     3           Plus           SMS                  13          11.75
     3           Plus           Voz                  18          59.06
     3           Prepago        Datos                 7          14.50
     3           Prepago        SMS                   7           6.82
     3           Prepago        Voz                  16          24.77
     4  Este                                         38          86.94
     4  Norte             


NOTE: PROC MEANS
NOTE: Output dataset work.cube_hier has 22 observations and 6 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_hier

NOTE: PROC PRINT completed: 22 observations printed, 6 variables


## Paso 4 - Proyección ponderada, exploración regional y triage de fugas

Tres lecturas que un equipo de aseguramiento de ingresos realmente ejecuta contra el cubo:

- **Proyección ponderada.** Adjuntar `WEIGHT=subscriber_wt` a un resumen `region*plan_tier` reporta el ingreso escalado a la base completa de suscriptores que representa la muestra, en lugar de las filas muestreadas sin ajustar.
- **Exploración.** Filtrar el cubo NWAY a una región expande una sola rama de la jerarquía - aquí Sur - a su detalle por nivel y servicio.
- **Triage de fugas.** Ordenar las celdas por ingreso promedio por registro revela las celdas de mayor rendimiento; las celdas con frecuencia baja y alto rendimiento son exactamente lo que una auditoría examina en busca de ingresos mal tarificados o con fugas.

In [4]:
/* Ingreso ponderado proyectado a la base de suscriptores */
PROC summary DATOS=work.cdr_billing NWAY;
    CLASE region plan_tier;
    VAR revenue;
    PESO subscriber_wt;
    SALIDA out=work.cube_wt(ELIMINAR=_type_ _freq_)
           sum(revenue)=rev_weighted  n(revenue)=records;
EJECUTAR;

PROC PRINT DATOS=work.cube_wt ETIQUETA noobs;
    ETIQUETA region       = "Región"
          plan_tier    = "Nivel de Plan"
          rev_weighted = "Ingreso Ponderado (Proyectado)"
          records      = "Registros";
    TÍTULO "Ingreso ponderado por region * nivel de plan (proyectado a la base de suscriptores)";
    FORMATO rev_weighted comma10.2;
EJECUTAR;

/* Explorar la rama de la region Sur del cubo */
PROC PRINT DATOS=work.cube_nway ETIQUETA noobs;
    DONDE region = "Sur";
    ETIQUETA plan_tier = "Nivel de Plan"
          call_type = "Tipo de Llamada"
          rev_total = "Ingreso Total"
          rev_mean  = "Ingreso Promedio";
    TÍTULO "Desglose de Sur: celdas de ingreso por nivel y tipo de llamada";
    VAR plan_tier call_type _freq_ rev_total rev_mean;
    FORMATO rev_total rev_mean comma10.2;
EJECUTAR;

/* Clasificar celdas por rendimiento por registro para triage de fugas */
PROC SORT DATOS=work.cube_nway out=work.cube_ranked;
    POR DESCENDENTE rev_mean;
EJECUTAR;

PROC PRINT DATOS=work.cube_ranked(obs=6) ETIQUETA noobs;
    ETIQUETA region    = "Región"
          plan_tier = "Nivel de Plan"
          call_type = "Tipo de Llamada"
          rev_mean  = "Ingreso Promedio"
          rev_max   = "Ingreso Máximo";
    TÍTULO "Celdas de mayor ingreso promedio (rendimiento por registro)";
    VAR region plan_tier call_type _freq_ rev_mean rev_max;
    FORMATO rev_mean rev_max comma10.2;
EJECUTAR;


                  Ingreso ponderado por region * nivel de plan (proyectado a la base de suscriptores)                   

 Región  Nivel de Plan  Ingreso Ponderado (Proyectado)  Registros
Este     Básico                                  50.85         15
Este     Plus                                    59.59         12
Este     Prepago                                 29.77         11
Norte    Básico                                  18.30          7
Norte    Plus                                    22.42          7
Norte    Prepago                                 20.56          9
Sur      Básico                                  58.63         14
Sur      Plus                                    56.29         15
Sur      Prepago                                 27.77         10

                             Desglose de Sur: celdas de ingreso por nivel y tipo de llamada                             

Nivel de Plan  Tipo de Llamada  _FREQ_  Ingreso Total  Ingreso Promedio
Básico         Datos   


NOTE: PROC MEANS
NOTE: Output dataset work.cube_wt has 9 observations and 4 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_wt

NOTE: PROC PRINT completed: 9 observations printed, 4 variables
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 9 observations printed, 5 variables
NOTE: PROC SORT data=work.cube_nway

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 25 rows from work.cube_nway.
NOTE: Wrote work.cube_ranked (25 rows, 9 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=work.cube_ranked

NOTE: PROC PRINT completed: 6 observations printed, 6 variables


## Interpretando los resultados

El cubo resumen convierte 100 filas originales de suscriptor-día en un conjunto compacto de celdas pre-agregadas que responden preguntas de exploración al instante, sin volver a recorrer la tabla de hechos:

- **Dónde vive el ingreso.** El marginal de `region` (`_TYPE_=4`) muestra que Sur liquidó \$97.44 y Este \$86.94 - juntos el 83% de los \$222.78 del mes - mientras que Norte contribuyó \$38.40. Dentro del subcubo `call_type*plan_tier` (`_TYPE_=3`), la Voz de nivel Plus es la celda individual más rica con \$59.06 en 18 registros, seguida de la Voz de nivel Básico con \$42.33.
- **Proyección ponderada.** Aplicar el peso de encuesta reordena la clasificación hacia los planes cuyos suscriptores tienen más peso: Este-Plus (\$59.59) y Sur-Básico (\$58.63) lideran el ingreso proyectado `region*plan_tier`, un panorama diferente al de los totales regionales sin ponderar y un recordatorio de reportar el ingreso proyectado en lugar del muestreado al dimensionar la exposición.
- **Rendimiento por registro y triage de fugas.** Clasificar las celdas NWAY por `rev_mean` coloca las celdas de datos móviles en la parte superior - Norte-Básico-Datos (\$7.87/registro) y Sur-Plus-Datos (\$5.96/registro) - confirmando que el uso intensivo de datos impulsa el mayor ingreso por registro. Los líderes de baja frecuencia (uno o dos registros) son precisamente las celdas para las que un analista de aseguramiento de ingresos extraería los registros de llamadas subyacentes, para confirmar que el cargo elevado está correctamente tarificado y no es un error.

Para un equipo de aseguramiento de ingresos, este cubo es la base de los paneles de varianza: comparar el ingreso liquidado contra el ingreso esperado de la tarifa por celda (región, nivel, tipo de llamada), y las celdas cuyo promedio o total ponderado se desvíen más se convierten en los casos de fuga que vale la pena auditar. Como toda la estructura es un conjunto de datos SAS ordinario, el cubo del próximo mes se puede unir, restar o combinar con una tarifa usando las mismas herramientas de Base SAS.